# Treinamento com Precisão Mista (AMP)

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Precisão mista guarda os parâmetros e cópia mestre em FP32 mas roda forward/backward em FP16/BF16. Com `autocast` + `GradScaler` você ganha ~2× de velocidade e metade da memória em GPUs modernas.


## Formulação Matemática

A cada passo:

1. $z = \text{autocast}(\text{forward}(x))$ em FP16/BF16.
2. Loss escalada: $\tilde L = s\,L$ para tirar os gradientes FP16 do underflow.
3. Unscale + step na cópia mestre FP32.


## Implementação


In [ ]:
import torch
import torch.nn as nn


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')

def step(x, y):
    with torch.cuda.amp.autocast(enabled=device == 'cuda'):
        logits = model(x)
        loss = nn.functional.cross_entropy(logits, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad()
    return loss.item()


## Experimento


In [ ]:
x = torch.randn(64, 256, device=device)
y = torch.randint(0, 10, (64,), device=device)
print('loss step 0:', step(x, y))
print('loss step 1:', step(x, y))


## Discussão

- BF16 (GPUs Ampere+) muitas vezes dispensa loss scaling porque o range de expoente bate com FP32.
- Desabilite autocast em operações numericamente sensíveis (reduções de layer-norm, softmax com logits extremos).
- Sempre cheque se o loss é finito — NaNs são o modo de falha clássico do AMP.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
